# Colab ViT-S/16 DINO: vit_s16_covidqu_syn

This notebook runs only `vit_s16_covidqu_syn`. It keeps pretraining and fine-tuning in separate cells so you can resume pretraining in epoch chunks.


## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone or Pull Repository

In [2]:
from pathlib import Path

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

!git rev-parse --short HEAD

/content
Cloning into 'contrastive-synthesis-medcls_CVProject'...
remote: Enumerating objects: 21515, done.
remote: Counting objects: 100% (242/242), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 21515 (delta 164), reused 167 (delta 103), pack-reused 21273 (from 3)
Receiving objects: 100% (21515/21515), 620.34 MiB | 19.56 MiB/s, done.
Resolving deltas: 100% (193/193), done.
Updating files: 100% (21263/21263), done.
/content/contrastive-synthesis-medcls_CVProject
4dfa0597


## 3. Install Minimal Dependencies

In [3]:
import importlib.util
import subprocess
import sys

packages = {
    'timm': 'timm',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'pandas': 'pandas',
    'PIL': 'Pillow',
}

to_install = []
for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        to_install.append(package_name)

if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
else:
    print('All required packages already installed.')

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi

All required packages already installed.
Torch: 2.11.0+cu128
CUDA available: True
Fri Jun  5 14:00:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |               

## 4. Editable Paths and Run Flags

`PRETRAIN_EPOCH_OVERRIDE` is the total target DINO epoch count. If a `last_dino_checkpoint.pth` already exists in the output folder, pretraining resumes up to this total.

`LOCAL_CROPS_NUMBER=4` uses DINO multi-crop. If a `timm` version still rejects 96x96 crops, set it to `0` and rerun pretraining.


In [4]:
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')
OUTPUT_ROOT = Path('/content/drive/MyDrive/medcls_cvproject/results/experiments')
SYNTHETIC_MANIFEST = Path('/content/drive/MyDrive/medcls_cvproject/data/manifests/synthetic_dcgan.csv')
REAL_UNLABELED_DIR = REPO_ROOT / 'data/processed/unlabelled_16934'

RUN_VIT_COVIDQU = False
RUN_VIT_IMAGENET_COVIDQU = False
RUN_VIT_COVIDQU_SYN = True
RUN_VIT_IMAGENET_COVIDQU_SYN = False

# Run pretraining in small chunks by increasing this total target: 10, 20, 30, ...
RUN_PRETRAIN = True
RUN_FINETUNE = True  # Set True only after the chosen pretraining target is finished.
PRETRAIN_EPOCH_OVERRIDE = 100
FINETUNE_EPOCH_OVERRIDE = None
LOCAL_CROPS_NUMBER = 4  # DINO default here. If timm crop-size errors persist, set this to 0.

pretrain_epoch_arg = '' if PRETRAIN_EPOCH_OVERRIDE is None else f'--epochs {PRETRAIN_EPOCH_OVERRIDE}'
local_crops_arg = f'--local-crops-number {LOCAL_CROPS_NUMBER}'
finetune_epoch_arg = '' if FINETUNE_EPOCH_OVERRIDE is None else f'--epochs {FINETUNE_EPOCH_OVERRIDE}'

%cd {REPO_ROOT}
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('SYNTHETIC_MANIFEST:', SYNTHETIC_MANIFEST, SYNTHETIC_MANIFEST.exists())
print('REAL_UNLABELED_DIR:', REAL_UNLABELED_DIR, REAL_UNLABELED_DIR.exists())
print('RUN_PRETRAIN:', RUN_PRETRAIN)
print('RUN_FINETUNE:', RUN_FINETUNE)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('local_crops_arg:', local_crops_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)

/content/contrastive-synthesis-medcls_CVProject
OUTPUT_ROOT: /content/drive/MyDrive/medcls_cvproject/results/experiments
SYNTHETIC_MANIFEST: /content/drive/MyDrive/medcls_cvproject/data/manifests/synthetic_dcgan.csv True
REAL_UNLABELED_DIR: /content/contrastive-synthesis-medcls_CVProject/data/processed/unlabelled_16934 True
RUN_PRETRAIN: True
RUN_FINETUNE: True
pretrain_epoch_arg: --epochs 100
local_crops_arg: --local-crops-number 4
finetune_epoch_arg: 


## 5. Verify Inputs and Scripts

In [5]:
!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_dino_vit.py scripts/run_classification_vit.py
!python scripts/run_dino_vit.py --help | grep resume || true
!python scripts/run_classification_vit.py --help | grep pretrained || true


Experiment Input Check Report
[PASS] common config
  - loaded configs/experiments/common.yaml
[PASS] fixed supervised manifests
  - train: {'COVID': 578, 'Lung_Opacity': 961, 'Viral_Pneumonia': 215, 'Normal': 1630} total=3384
  - val: {'COVID': 72, 'Lung_Opacity': 120, 'Viral_Pneumonia': 26, 'Normal': 203} total=421
  - test: {'COVID': 73, 'Lung_Opacity': 121, 'Viral_Pneumonia': 28, 'Normal': 205} total=427
[PASS] experiment config files
[PASS] resnet18_covidqu
  - planned output_dir: results/experiments/resnet18_covidqu
[PASS] resnet18_covidqu_syn
  - synthetic_dcgan: {'COVID': 1000, 'Lung_Opacity': 1000, 'Viral_Pneumonia': 1000, 'Normal': 1000} total=4000
  - planned output_dir: results/experiments/resnet18_covidqu_syn
[PASS] resnet18_imagenet
  - no contrastive pretraining data required
  - planned output_dir: results/experiments/resnet18_imagenet
[PASS] resnet18_imagenet_covidqu
  - planned output_dir: results/experiments/resnet18_imagenet_covidqu
[PASS] resnet18_imagenet_covidqu_

## 6. Experiment: vit_s16_covidqu_syn

DINO pretraining from random initialization on Stage 1 DCGAN synthetic images, then supervised fine-tuning on real labeled manifests.

Run the pretraining cell repeatedly by increasing `PRETRAIN_EPOCH_OVERRIDE` from 10 to 20, 30, and so on. Run the fine-tuning cell only after pretraining reaches the target you want to report.

### 6a. Pretrain Only: vit_s16_covidqu_syn

In [6]:
EXP = 'vit_s16_covidqu_syn'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_dino_teacher.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_dino_checkpoint.pth'

print('OUT:', OUT)
print('CKPT:', CKPT, 'exists=', CKPT.exists())
print('RESUME_CKPT:', RESUME_CKPT, 'exists=', RESUME_CKPT.exists())

if RUN_VIT_COVIDQU_SYN and RUN_PRETRAIN:
    !python scripts/run_dino_vit.py \
      --config configs/experiments/vit_s16/covidqu_syn.yaml \
      --synthetic-manifest "{SYNTHETIC_MANIFEST}" \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {local_crops_arg} \
      {pretrain_epoch_arg}
else:
    print('Skipping pretrain', EXP)


OUT: /content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn
CKPT: /content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn/pretrain/checkpoints/best_dino_teacher.pth exists= True
RESUME_CKPT: /content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn/pretrain/checkpoints/last_dino_checkpoint.pth exists= True
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Resuming DINO from /content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn/pretrain/checkpoints/last_dino_checkpoint.pth at epoch 39
Epoch 40/100 dino_loss=7.9696
Epoch 41/100 dino_loss=7.9479
Epoch 42/100 dino_loss=7.9286
Epoch 43/100 dino_loss=7.9085
Epoch 44/100 dino_loss=7.9014
Epoch 45/100 dino_loss=7.8750
Epoch 46/100 dino_loss=7.8612
Epoch 47/100 dino_loss=7

### 6b. Fine-Tune Only: vit_s16_covidqu_syn

In [7]:
EXP = 'vit_s16_covidqu_syn'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_dino_teacher.pth'

if RUN_VIT_COVIDQU_SYN and RUN_FINETUNE:
    if not CKPT.exists():
        raise FileNotFoundError(f'DINO checkpoint not found: {CKPT}. Finish pretraining first.')
    !python scripts/run_classification_vit.py \
      --config configs/experiments/vit_s16/covidqu_syn.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {finetune_epoch_arg}
else:
    print('Skipping finetune', EXP)


Missing keys after DINO teacher load: ['head.weight', 'head.bias']
Epoch 1/50 train_loss=0.9993 val_loss=0.8246 val_acc=0.6888 val_f1_macro=0.6114
Epoch 2/50 train_loss=0.7777 val_loss=0.7486 val_acc=0.7007 val_f1_macro=0.6446
Epoch 3/50 train_loss=0.6793 val_loss=0.7148 val_acc=0.7150 val_f1_macro=0.6684
Epoch 4/50 train_loss=0.6223 val_loss=0.6613 val_acc=0.7387 val_f1_macro=0.7152
Epoch 5/50 train_loss=0.5647 val_loss=0.6139 val_acc=0.7601 val_f1_macro=0.7488
Epoch 6/50 train_loss=0.5225 val_loss=0.5956 val_acc=0.7767 val_f1_macro=0.7594
Epoch 7/50 train_loss=0.4885 val_loss=0.6019 val_acc=0.7553 val_f1_macro=0.7447
Epoch 8/50 train_loss=0.4714 val_loss=0.6058 val_acc=0.7506 val_f1_macro=0.7287
Epoch 9/50 train_loss=0.4519 val_loss=0.6018 val_acc=0.7577 val_f1_macro=0.7573
Epoch 10/50 train_loss=0.4339 val_loss=0.5804 val_acc=0.7720 val_f1_macro=0.7620
Epoch 11/50 train_loss=0.4110 val_loss=0.5045 val_acc=0.7957 val_f1_macro=0.7917
Epoch 12/50 train_loss=0.3781 val_loss=0.5239 val_a

## 7. Display Result


In [8]:
import json
import pandas as pd

EXP = 'vit_s16_covidqu_syn'
metrics_path = OUTPUT_ROOT / EXP / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame([{**{'experiment_id': EXP}, **metrics}]))
else:
    print('No metrics found yet:', metrics_path)

!find "{OUTPUT_ROOT}/vit_s16_covidqu_syn" -maxdepth 4 -type f \( -name 'metrics.json' -o -name 'best_dino_teacher.pth' -o -name 'last_dino_checkpoint.pth' -o -name 'confusion_matrix.png' -o -name 'classification_report.csv' \) | sort


,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,vit_s16_covidqu_syn,0.824356,0.824655,0.805343,0.812589,0.825582,0.824356,0.822312,31,0.820674


/content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn/classification_report.csv
/content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn/confusion_matrix.png
/content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn/metrics.json
/content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn/pretrain/checkpoints/best_dino_teacher.pth
/content/drive/MyDrive/medcls_cvproject/results/experiments/vit_s16_covidqu_syn/pretrain/checkpoints/last_dino_checkpoint.pth
